In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
import plotly.express as px
from plotly.offline import init_notebook_mode
import re
import nltk
from nltk.corpus import stopwords
from tqdm import tqdm
from nltk.stem import WordNetLemmatizer
import spacy

nltk.download("omw-1.4", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
tqdm.pandas()
spacy_eng = spacy.load("en_core_web_sm")
lem = WordNetLemmatizer()
init_notebook_mode(connected=True)
sns.set_style("darkgrid")
plt.rcParams["figure.figsize"] = (20,8)
plt.rcParams["font.size"] = 18

In [ ]:
data = pd.read_csv('train.csv')
data.head(15)

In [ ]:
data.isnull().sum()

In [ ]:
data.dropna(inplace=True)

In [ ]:
fig = px.pie(data, values='id', names='is_duplicate', title='Duplicate And Non Duplicate Questions Percentage')
fig.show()

In [ ]:
def text_cleaning(words):
    question = re.sub(r'\s+\n+', ' ', words)  # remove extra spaces and newlines
    question = re.sub(r'[^a-zA-Z0-9]', ' ', question)
    question = question.lower()
    return question

In [ ]:
data['question1_clean'] = data['question1'].progress_apply(text_cleaning)
data['question2_clean'] = data['question2'].progress_apply(text_cleaning)
data

In [ ]:
data['question1_length'] = data['question1_clean'].apply(lambda x: len(x.split()))
data['question2_length'] = data['question2_clean'].apply(lambda x: len(x.split()))

In [ ]:
px.histogram(data, x='question1_length', color='is_duplicate', title='Question 1 Length Distribution Plot')

In [ ]:
px.histogram(data, x='question2_length', color='is_duplicate', title='Question 2 Length Distribution Plot')

In [ ]:
questions1 = data['question1_clean'].tolist()
questions2 = data['question2_clean'].tolist()

In [ ]:
wordcloud = WordCloud(max_words=1500, background_color='black').generate(" ".join(questions1))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Words From Question1')
plt.axis('off')
plt.show()

In [ ]:
wordcloud = WordCloud(max_words=1500, background_color='black').generate(" ".join(questions2))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Words From Question2')
plt.axis('off')
plt.show()

In [ ]:
data['question1_length'].describe()

In [ ]:
data['question2_length'].describe()

In [ ]:
import tensorflow as tf
import tf_keras
from tf_keras.preprocessing.text import Tokenizer
from tf_keras.preprocessing.sequence import pad_sequences
from tf_keras.utils import to_categorical
from tf_keras.models import Sequential, Model
from tf_keras import layers
from tf_keras.layers import (Embedding, Layer, Dense, Dropout, MultiHeadAttention,
                              LayerNormalization, Input, GlobalAveragePooling1D,
                              LSTM, Bidirectional)
from tf_keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from transformers import (AutoTokenizer, DataCollatorWithPadding, TFAutoModel,
                           DistilBertConfig, TFDistilBertModel, BertConfig,
                           TFBertModel, TFRobertaModel)
from datasets import load_dataset

In [ ]:
model_checkpoint = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
def encode_text(text, tokenizer):
    encoded = tokenizer.batch_encode_plus(text, add_special_tokens=True, max_length=50,padding='max_length', truncation=True, return_attention_mask=True, return_tensors='tf')

    input_ids = np.array(encoded['input_ids'], dtype='int32')
    attention_masks = np.array(encoded['attention_mask'], dtype='int32')
    return {'input_ids': input_ids, 'attention_masks': attention_masks}

In [ ]:
n_sample = min(400000, len(data))
data = data.sample(n_sample)
train = data.iloc[:int(n_sample * 0.80), :]
val = data.iloc[int(n_sample * 0.80):, :]

question1_train = encode_text(train["question1_clean"].tolist(), tokenizer)
question2_train = encode_text(train["question2_clean"].tolist(), tokenizer)
question1_val = encode_text(val["question1_clean"].tolist(), tokenizer)
question2_val = encode_text(val["question2_clean"].tolist(), tokenizer)

y_train = train["is_duplicate"].values
y_val = val["is_duplicate"].values

In [ ]:
# Make data with 3 questions where 3rd question is entire random and not related to other questions

In [ ]:
duplicate_data = data.query("is_duplicate == 1").copy()
duplicate_data

In [ ]:
duplicate_data.shape

In [ ]:
non_duplicate_data = data.query('is_duplicate == 0')
question3_col = list(non_duplicate_data[:duplicate_data.shape[0]]['question1'])
question3_col

In [ ]:
duplicate_data['question3'] = question3_col

In [ ]:
duplicate_data.head(15)

In [ ]:
duplicate_data['question3_clean'] = duplicate_data['question3'].progress_apply(text_cleaning)

In [ ]:
duplicate_data.head(25)

In [ ]:
train = duplicate_data.iloc[:int(duplicate_data.shape[0]*0.80),:]
val = duplicate_data.iloc[int(duplicate_data.shape[0]*0.80):,:]

question1_train = encode_text(train['question1_clean'].tolist(), tokenizer)
question2_train = encode_text(train['question2_clean'].tolist(), tokenizer)
question3_train = encode_text(train['question3_clean'].tolist(), tokenizer)

question1_val = encode_text(val['question1_clean'].tolist(), tokenizer)
question2_val = encode_text(val['question2_clean'].tolist(), tokenizer)
question3_val = encode_text(val['question3_clean'].tolist(), tokenizer)

In [ ]:
class DistanceLayer(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, anchor, positive, negative):
        positive_distance = tf.reduce_sum(tf.square(anchor - positive), -1)
        negative_distance = tf.reduce_sum(tf.square(anchor - negative), -1)
        return positive_distance, negative_distance

In [ ]:
from tf_keras import metrics

In [ ]:
class SiameseModel(Model):

    def __init__(self, siamese_network, margin=0.5):
        super(SiameseModel, self).__init__()
        self.siamese_network = siamese_network
        self.margin = margin
        self.loss_tracker = metrics.Mean(name="loss")

    def call(self, inputs):
        return self.siamese_network(inputs)

    def train_step(self, data):

        with tf.GradientTape() as tape:
            loss = self._compute_loss(data)

        gradients = tape.gradient(loss, self.siamese_network.trainable_weights)
        self.optimizer.apply_gradients(
            zip(gradients, self.siamese_network.trainable_weights)
        )
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def test_step(self, data):
        loss = self._compute_loss(data)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def _compute_loss(self, data):
        ap_distance, an_distance = self.siamese_network(data)
        loss = ap_distance - an_distance
        loss = tf.maximum(loss + self.margin, 0.0)
        return loss

    @property
    def metrics(self):
        return [self.loss_tracker]


In [ ]:
transformer_model = TFBertModel.from_pretrained(model_checkpoint)

input_ids_in1   = Input(shape=(50,), name='input_ids1',     dtype='int32')
input_masks_in1 = Input(shape=(50,), name='attention_mask1', dtype='int32')

anchor_input   = Input(name='anchor_ids',    shape=(50,), dtype='int32')
anchor_masks   = Input(name='anchor_mask',   shape=(50,), dtype='int32')
positive_input = Input(name='positive_ids',  shape=(50,), dtype='int32')
positive_masks = Input(name='positive_mask', shape=(50,), dtype='int32')
negative_input = Input(name='negative_ids',  shape=(50,), dtype='int32')
negative_masks = Input(name='negative_mask', shape=(50,), dtype='int32')

embedding_layer = transformer_model(input_ids_in1, attention_mask=input_masks_in1).last_hidden_state

average = GlobalAveragePooling1D()(embedding_layer)
embeds  = Dense(512, activation='relu')(average)
embeddings = Model(inputs=[input_ids_in1, input_masks_in1], outputs=embeds)

for layer in embeddings.layers[:-1]:
    layer.trainable = False

embeds1 = embeddings([anchor_input,   anchor_masks])
embeds2 = embeddings([positive_input, positive_masks])
embeds3 = embeddings([negative_input, negative_masks])

distances = DistanceLayer()(embeds1, embeds2, embeds3)

siamese_network = Model(
    inputs=[anchor_input, anchor_masks, positive_input, positive_masks,
            negative_input, negative_masks],
    outputs=distances
)

siamese_model = SiameseModel(siamese_network)
siamese_model.compile(optimizer=tf_keras.optimizers.Adam(0.00001))

# tf.data.Dataset ensures Keras batches along axis-0 (not len(tuple)=6)
BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices((
    np.asarray(question1_train['input_ids']),
    np.asarray(question1_train['attention_masks']),
    np.asarray(question2_train['input_ids']),
    np.asarray(question2_train['attention_masks']),
    np.asarray(question3_train['input_ids']),
    np.asarray(question3_train['attention_masks'])
)).batch(BATCH_SIZE)

val_dataset = tf.data.Dataset.from_tensor_slices((
    np.asarray(question1_val['input_ids']),
    np.asarray(question1_val['attention_masks']),
    np.asarray(question2_val['input_ids']),
    np.asarray(question2_val['attention_masks']),
    np.asarray(question3_val['input_ids']),
    np.asarray(question3_val['attention_masks'])
)).batch(BATCH_SIZE)

history = siamese_model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset
)

In [ ]:
plt.figure(figsize=(20,8))
plt.plot(history.history["loss"])
if "val_loss" in history.history:
    plt.plot(history.history["val_loss"])
    plt.legend(["train", "val"], loc="upper left")
else:
    plt.legend(["train"], loc="upper left")
plt.title("model loss")
plt.ylabel("loss")
plt.xlabel("epoch")
plt.show()

In [ ]:
def get_cosine_similarity(sentence1, sentence2):
    f1 = text_cleaning(sentence1)
    f1 = encode_text([f1], tokenizer)
    f2 = text_cleaning(sentence2)
    f2 = encode_text([f2], tokenizer)

    f1_inputs = np.array(f1['input_ids'])
    f1_masks = np.array(f1['attention_masks'])
    f2_inputs = np.array(f2['input_ids'])
    f2_masks = np.array(f2['attention_masks'])

    embeddings1 = embeddings([f1_inputs, f1_masks])
    embeddings2 = embeddings([f2_inputs, f2_masks])

    cosine_similarity = metrics.CosineSimilarity()
    return cosine_similarity(embeddings1, embeddings2).numpy()

In [ ]:
sentence1 = 'Is Earth circle in shape ?'
sentence2 = 'Should I learn python as it is very popular ?'
get_cosine_similarity(sentence1,sentence2)

In [ ]:
sentence1 = 'Python is one of the most popular programming language out there'
sentence2 = 'Should I learn python programming as it is very popular ?'
get_cosine_similarity(sentence1,sentence2)

In [ ]:
sentence1 = 'Which GPU gives a better performance NVIDIA or AMD ?'
sentence2 = 'My friend has a NVIDIA GPU, and he suggests that it gives a very smooth gaming performance'
get_cosine_similarity(sentence1,sentence2)